In [34]:
# import key libraries

# import Python libraries
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

# import PyTorch libraries
import torch
import torch.nn.functional as F

# import Hugging Face libraries
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# import Zephyr Tokenizer
zephyr_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")

# Install bitsandbytes
!pip install -U bitsandbytes

# set BitsAndBytesConfig object
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# import the model
zephyr_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceH4/zephyr-7b-beta", quantization_config=quantization_config, device_map="auto"
)

zephyr_model.eval()
zephyr_model.to(device);

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [6]:
# Find the number of paarmeters in the model
num_param = sum(p.numel() for p in zephyr_model.parameters())

# calculate the number of paarmeters manually
num_param_manual = 0

num_trainable_params = sum(p.numel() for p in zephyr_model.parameters() if p.requires_grad)
num_non_trainable_params = num_param - num_trainable_params

print(f"Number of parameters in the model: {num_param:,} ({num_param/10**9:.2f}B)")
print(f"Number of trainable paremeters in the model: {num_trainable_params:,}")

print(f"Number of non-trainable parameters in the model: {num_non_trainable_params:,}")


Number of parameters in the model: 3,752,071,168 (3.75B)
Number of trainable paremeters in the model: 262,410,240
Number of non-trainable parameters in the model: 3,489,660,928


In [9]:
# import HellaSwag dataset
!pip install --upgrade datasets

dataset = load_dataset("Rowan/hellaswag", split="validation")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/24.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/6.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/6.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39905 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10042 [00:00<?, ? examples/s]

In [44]:
# Function to test one HellSwag sample
def test_hellaswag_sample(sample, model, tokenizer):

  # find context/ sequence length
  context = sample['ctx']
  context_len = len(zephyr_tokenizer.encode(context))

  # initialize an array with the length of all endings
  all_endings = len(sample['endings'])
  sum_sequences = np.zeros(all_endings)

  # loop over context endings, create prompts, get logits and sum over probability scores

  for opti in range(all_endings):

    # prompts and their lengths
    prompt = f'{context} {sample["endings"][opti]}'
    prompt_tokens = zephyr_tokenizer.encode(prompt, return_tensors='pt')
    prompt_len = len(prompt_tokens[0])

    # forward pass through the model
    with torch.no_grad():
      output = model(prompt_tokens.to(device))
      logits = output.logits

    # convert to log-probabilities
    log_probs = F.log_softmax(logits, dim=-1)

    # get the prediction of log probabilities for each token
    # Corrected loop range to prevent IndexError and typo 'sm_seq'
    sum_seq = np.array([log_probs[0,i, prompt_tokens[0][i+1]].item() for i in range(0,prompt_len - 1)])

    sum_sequences[opti] = np.sum(sum_seq)

  # Sum log probabilities to get total log-likelihood
  return sum_sequences, int(sample['label'])

In [48]:
# test with one sample
log_likelihood, answer = test_hellaswag_sample(dataset[42], zephyr_model,zephyr_tokenizer)

print('Correct' if answer == np.argmax(log_likelihood) else 'Incorrect')

Correct


In [53]:
# Import gpt2-small model
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

#

In [56]:
# Train the Zephyr model with the complete dataset and find accuracy
num_samples = 500

zephyr_acc = np.zeros(num_samples)

for sampli in tqdm(range(num_samples),desc='Sample Number'):
  log_likelihood, answer = test_hellaswag_sample(dataset[sampli], zephyr_model,zephyr_tokenizer)
  zephyr_acc[sampli] = 1 if answer == np.argmax(log_likelihood) else 0

zephyr_mean_acc = np.mean(zephyr_acc)

print(f'Accuracy for Zephyr: {100*zephyr_mean_acc:.2f}%')


Sample Number: 100%|██████████| 500/500 [03:13<00:00,  2.59it/s]

Accuracy for Zephyr: 51.20%


In [55]:
# Train the GPT-2 model with the complete dataset and find accuracy
num_samples = 500

gpt2_acc = np.zeros(num_samples)

for sampli in tqdm(range(num_samples),desc='Sample Number'):
  log_likelihood, answer = test_hellaswag_sample(dataset[sampli], gpt2_model,gpt2_tokenizer)
  gpt2_acc[sampli] = 1 if answer == np.argmax(log_likelihood) else 0

gpt2_mean_acc = np.mean(gpt2_acc)

print(f'Accuracy for GPT2: {100*gpt2_mean_acc:.2f}%')


Sample Number: 100%|██████████| 500/500 [00:22<00:00, 22.20it/s]

Accuracy for GPT2: 27.80%


In [68]:
# Find the indices for which Zephyr is correct and GPT2 is wrong
indices_zephyr_correct_gpt2_wrong = np.where((zephyr_acc == 1) & (gpt2_acc == 0))
sample_index = indices_zephyr_correct_gpt2_wrong[0][-1]
print(dataset[sample_index]['ctx'])
print(dataset[sample_index]['endings'][int(dataset[sample_index]['label'])])

The man roll the barbel on the floor. the man
lifted the barbel up to his chest.
